In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('train_fe_v3.csv')
test  = pd.read_csv('test_fe_v3.csv')

TARGET = '임신 성공 여부'
ID_COL = 'ID'

print('Train:', train.shape, '/ Test:', test.shape)

Train: (256351, 192) / Test: (90067, 191)


In [3]:
# Feature_refinement 재적용 + 피처 준비
ZERO_IMP_COLS = [
    '배아생성_제로_플래그','배아생성_실패_플래그','난자수집_실패_플래그',
    '난자저장용_포함','배반포이식_여부','불임 원인 - 여성 요인',
    'IVF_배반포_조합','불임 원인 - 정자 면역학적 요인','high_resp_freezeall',
]
drop_cols = [c for c in ZERO_IMP_COLS if c in train.columns]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

for df in [train, test]:
    emb_n   = df['이식된 배아 수'].fillna(0)
    emb_log = df['이식된 배아 수_log'].fillna(0)
    age     = df['나이_순서'].fillna(0)
    quality = df['배아품질_종합점수'].fillna(0)
    is_ivf  = (df['시술 유형'] == 'IVF').astype(int)
    is_tx   = df['이식배아_있음'].fillna(0)
    tx_day  = df['배아 이식 경과일'].replace(-1, np.nan).fillna(0)

    df['이식배아수_나이_교호']     = emb_n * age
    df['이식배아log_나이_교호']    = emb_log * age
    df['이식배아수_품질_교호']     = emb_n * quality
    df['이식배아log_품질_교호']    = emb_log * quality
    df['이식배아수_IVF_교호']      = emb_n * is_ivf
    df['이식성공_품질_교호']       = is_tx * quality
    df['고령자가난자_품질_교호']   = df['자가난자_고령_조합'].fillna(0) * quality
    df['고위험3중_이식배아_교호']  = df['고위험_3중_조합'].fillna(0) * emb_n
    df['이식경과일_이식배아_교호'] = tx_day * emb_n
    df['이식배아수_출산경험_교호'] = emb_n * df['출산경험_있음'].fillna(0)
    df['나이_저장비율_교호']       = age * df['저장_비율'].fillna(0)
    df['잔여배아_품질_교호']       = df['잔여배아_수'].fillna(0) * quality
    df['이식배아수_제곱']          = emb_n ** 2
    df['이식배아_총배아_곱']       = emb_n * df['총 생성 배아 수'].fillna(0)
    df['단일배아이식_여부']        = (emb_n == 1).astype(int)
    df['2개배아이식_여부']         = (emb_n == 2).astype(int)

print('피처 정제 완료')

피처 정제 완료


In [4]:
feat_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]

X      = train[feat_cols].copy()
y      = train[TARGET].copy()
X_test = test[feat_cols].copy()

# 범주형 탐지
cat_features = [
    c for c in feat_cols
    if X[c].dtype == 'object' or pd.api.types.is_string_dtype(X[c])
]
for col in cat_features:
    X[col]      = X[col].astype(str).fillna('missing')
    X_test[col] = X_test[col].astype(str).fillna('missing')

cat_feature_indices = [X.columns.get_loc(c) for c in cat_features]

# LightGBM용
X_lgb      = X.copy()
X_test_lgb = X_test.copy()
for col in cat_features:
    X_lgb[col]      = X_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

# XGBoost용: 범주형 → 레이블 인코딩
from sklearn.preprocessing import OrdinalEncoder
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_xgb      = X.copy()
X_test_xgb = X_test.copy()
X_xgb[cat_features]      = enc.fit_transform(X[cat_features])      # train으로만 fit
X_test_xgb[cat_features] = enc.transform(X_test[cat_features])     # test는 transform
# 수치형 NaN → 중앙값 (XGBoost는 NaN 처리 가능하지만 명시적으로)
num_features = [c for c in feat_cols if c not in cat_features]
medians = X_xgb[num_features].median()   # train에서만 fit
X_xgb[num_features]      = X_xgb[num_features].fillna(medians)
X_test_xgb[num_features] = X_test_xgb[num_features].fillna(medians)

neg = (y == 0).sum()
pos = (y == 1).sum()
spw = neg / pos

print(f'피처: {len(feat_cols)}개 / 범주형: {len(cat_features)}개')
print(f'scale_pos_weight: {spw:.2f}')

피처: 197개 / 범주형: 21개
scale_pos_weight: 2.87


In [ ]:
# 1층: 3개 Base Model OOF 생성
N_SPLITS = 5
SEED     = 42
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# 파라미터
cat_params = {
    'iterations': 794, 'learning_rate': 0.030094391673729362,
    'depth': 7, 'l2_leaf_reg': 7.433949857398995,
    'bagging_temperature': 0.3980358270898108,
    'random_strength': 1.3075975210328405, 'border_count': 108,
    'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'scale_pos_weight': spw, 'random_seed': SEED,
    'early_stopping_rounds': 50, 'verbose': False, 'use_best_model': True,
}
lgb_params = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'num_leaves': 22, 'max_depth': 7, 'learning_rate': 0.02030926523583015,
    'n_estimators': 956, 'subsample': 0.9916893928795039,
    'colsample_bytree': 0.8760988294276582,
    'reg_alpha': 0.5158539052029842, 'reg_lambda': 0.05557192411355649,
    'min_child_samples': 98,
    'scale_pos_weight': spw, 'random_state': SEED, 'verbose': -1,
}
xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'auc',
    'max_depth': 7, 'learning_rate': 0.03,
    'n_estimators': 1000, 'subsample': 0.8,
    'colsample_bytree': 0.8, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'scale_pos_weight': spw, 'random_state': SEED,
    'tree_method': 'hist', 'verbosity': 0,
}

# OOF 배열
oof_cat  = np.zeros(len(X))
oof_lgb  = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
test_cat = np.zeros(len(X_test))
test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))

cat_scores, lgb_scores, xgb_scores = [], [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr  = X.iloc[tr_idx].reset_index(drop=True)
    X_val = X.iloc[val_idx].reset_index(drop=True)
    y_tr  = y.iloc[tr_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].reset_index(drop=True)

    # CatBoost
    cb = CatBoostClassifier(**cat_params)
    cb.fit(Pool(X_tr, y_tr, cat_features=cat_feature_indices),
           eval_set=Pool(X_val, y_val, cat_features=cat_feature_indices))
    p_cat = cb.predict_proba(Pool(X_val, cat_features=cat_feature_indices))[:, 1]
    oof_cat[val_idx]  = p_cat
    test_cat         += cb.predict_proba(Pool(X_test, cat_features=cat_feature_indices))[:, 1] / N_SPLITS
    cat_scores.append(roc_auc_score(y_val, p_cat))

    # LightGBM
    lb = lgb.LGBMClassifier(**lgb_params)
    lb.fit(X_lgb.iloc[tr_idx], y_tr,
           eval_set=[(X_lgb.iloc[val_idx], y_val)],
           callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    p_lgb = lb.predict_proba(X_lgb.iloc[val_idx])[:, 1]
    oof_lgb[val_idx]  = p_lgb
    test_lgb         += lb.predict_proba(X_test_lgb)[:, 1] / N_SPLITS
    lgb_scores.append(roc_auc_score(y_val, p_lgb))

    # XGBoost
    xb = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=50)
    xb.fit(X_xgb.iloc[tr_idx], y_tr,
           eval_set=[(X_xgb.iloc[val_idx], y_val)],
           early_stopping_rounds=50, verbose=False)
    p_xgb = xb.predict_proba(X_xgb.iloc[val_idx])[:, 1]
    oof_xgb[val_idx]  = p_xgb
    test_xgb         += xb.predict_proba(X_test_xgb)[:, 1] / N_SPLITS
    xgb_scores.append(roc_auc_score(y_val, p_xgb))

    print(f'Fold {fold+1} | CAT: {cat_scores[-1]:.5f} | LGB: {lgb_scores[-1]:.5f} | XGB: {xgb_scores[-1]:.5f}')

print(f'\n✅ CatBoost OOF AUC : {roc_auc_score(y, oof_cat):.5f}')
print(f'✅ LightGBM OOF AUC : {roc_auc_score(y, oof_lgb):.5f}')
print(f'✅ XGBoost  OOF AUC : {roc_auc_score(y, oof_xgb):.5f}')

TypeError: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'

In [ ]:
# 소프트 보팅 (3모델 최적 가중치 탐색)
best_auc_vote, best_w = 0.0, (0.5, 0.3, 0.2)
results = []

# 0.1 단위 그리드 탐색 (3모델 합 = 1)
for w1 in np.arange(0.0, 1.1, 0.1):
    for w2 in np.arange(0.0, 1.1 - w1, 0.1):
        w3 = round(1 - w1 - w2, 1)
        if w3 < 0:
            continue
        blended = oof_cat * w1 + oof_lgb * w2 + oof_xgb * w3
        auc = roc_auc_score(y, blended)
        results.append((round(w1,1), round(w2,1), round(w3,1), auc))
        if auc > best_auc_vote:
            best_auc_vote = auc
            best_w = (w1, w2, w3)

results_df = pd.DataFrame(results, columns=['w_cat','w_lgb','w_xgb','oof_auc'])
print('=== 소프트 보팅 상위 10개 ===')
display(results_df.sort_values('oof_auc', ascending=False).head(10))
print(f'\n✅ 소프트 보팅 최적 AUC: {best_auc_vote:.5f}')
print(f'   최적 가중치: CAT={best_w[0]:.1f}, LGB={best_w[1]:.1f}, XGB={best_w[2]:.1f}')

In [ ]:
# 2층 스태킹 (Ridge Logistic Regression)
# OOF 예측값을 메타 피처로
meta_train = np.column_stack([oof_cat, oof_lgb, oof_xgb])
meta_test  = np.column_stack([test_cat, test_lgb, test_xgb])

# Ridge Logistic: 단순하게 (메타 모델은 과적합 방지가 핵심)
scaler = StandardScaler()
meta_train_scaled = scaler.fit_transform(meta_train)  # train으로만 fit
meta_test_scaled  = scaler.transform(meta_test)

# C 값 탐색 (정규화 강도)
best_auc_stack, best_C = 0.0, 1.0
for C in [0.001, 0.01, 0.1, 1.0, 10.0]:
    meta_oof = np.zeros(len(y))
    for tr_idx, val_idx in StratifiedKFold(5, shuffle=True, random_state=SEED).split(meta_train_scaled, y):
        lr = LogisticRegression(C=C, max_iter=1000, random_state=SEED)
        lr.fit(meta_train_scaled[tr_idx], y.iloc[tr_idx])
        meta_oof[val_idx] = lr.predict_proba(meta_train_scaled[val_idx])[:, 1]
    auc = roc_auc_score(y, meta_oof)
    print(f'  C={C:.3f} | Meta AUC: {auc:.5f}')
    if auc > best_auc_stack:
        best_auc_stack = auc
        best_C = C

print(f'\n✅ 스태킹 최적 AUC: {best_auc_stack:.5f}  (C={best_C})')

# 최적 C로 전체 train 학습 → test 예측
meta_model = LogisticRegression(C=best_C, max_iter=1000, random_state=SEED)
meta_model.fit(meta_train_scaled, y)
test_stack = meta_model.predict_proba(meta_test_scaled)[:, 1]

In [ ]:
# 최종 비교 및 제출 파일 생성
# 소프트 보팅 test 예측
test_vote = test_cat * best_w[0] + test_lgb * best_w[1] + test_xgb * best_w[2]

print('=== 최종 성능 비교 ===')
print(f'  CatBoost 단독       : {roc_auc_score(y, oof_cat):.5f}')
print(f'  LightGBM 단독       : {roc_auc_score(y, oof_lgb):.5f}')
print(f'  XGBoost 단독        : {roc_auc_score(y, oof_xgb):.5f}')
print(f'  소프트 보팅 (3모델) : {best_auc_vote:.5f}')
print(f'  스태킹 (Ridge)      : {best_auc_stack:.5f}')
print(f'  이전 최고 (2모델)   : 0.74041')
print(f'  베이스라인          : 0.74008')

# 더 높은 것 선택
if best_auc_stack >= best_auc_vote:
    best_test_pred = test_stack
    best_method    = f'스태킹 (AUC={best_auc_stack:.5f})'
else:
    best_test_pred = test_vote
    best_method    = f'소프트보팅 (AUC={best_auc_vote:.5f})'

print(f'\n→ 제출 선택: {best_method}')

sample_sub = pd.read_csv('/Users/Jiyeon/Desktop/ftp/fertility-treatment-prediction/data/raw/sample_submission.csv')
submission = sample_sub.copy()
submission[sample_sub.columns[1]] = best_test_pred
submission.to_csv('submission_stacking.csv', index=False, encoding='utf-8-sig')
print('저장 완료: submission_stacking.csv')
display(submission.head())